# Other Visualizations: Model Evaluation Diagnostics

These visualizations are for comparing model predictions against ground truth.
They apply to any PDE-based or learned operator model, not specifically to barotropic data.

Requires a prediction field and a truth field of the same shape.

| # | Visualization | What it shows |
|---|---|---|
| 3.1 | Error Maps | Spatial error structure (absolute, signed, relative) |
| 3.2 | Power Spectrum Comparison | Spectral fidelity of predictions |
| 3.3 | PDE Residual Map | Physics consistency of model output |

---
## Setup: Synthetic Prediction Data

Since we do not have model predictions yet, we generate a synthetic prediction
by Gaussian-blurring the true vorticity field. This mimics the over-smoothing
bias common in learned operators.

In [ ]:
import matplotlib.pyplot as plt
from ml.data.isca_segment import read_segments
from ml.diagnostics.examples import synthetic_prediction
from ml.visualization.examples import plot_error_maps, plot_spectrum_comparison, plot_advection_residual

plt.rcParams["figure.dpi"] = 120
SIM_DIR = "../../output/barotropic_stirring-T85/simulations/0"
ds = read_segments(f"{SIM_DIR}/run*/atmos_daily.nc", decode_time=True)
VOR_TRUE, VOR_PRED = synthetic_prediction(ds)


In [ ]:
# One shared reader replaces the two inconsistent setup cells.


---
## 3.1 Error Maps

Three views of the error between predicted and true vorticity:

- **Absolute error** $|\hat{\zeta} - \zeta|$: Where is the error largest?
- **Signed error** $\hat{\zeta} - \zeta$: Over-prediction (red) vs under-prediction (blue).
- **Relative error** $|\hat{\zeta} - \zeta| / |\zeta|$: Error relative to local signal strength.

Always look at spatial error maps before relying on scalar metrics like RMSE --
a model with low RMSE might still have large localized errors.

In [ ]:
plot_error_maps(VOR_PRED, VOR_TRUE, ds.lon.values, ds.lat.values)


---
## 3.2 Power Spectrum Comparison

**Left:** Zonal power spectrum of both fields on log-log axes.
**Right:** Spectral ratio $P_{\text{pred}}(k) / P_{\text{true}}(k)$.

Ratio < 1 = prediction missing energy (over-smoothed).
Ratio > 1 = excess energy (noisy).
Ratio = 1 = spectrally accurate.

In [ ]:
plot_spectrum_comparison(VOR_PRED, VOR_TRUE, ds.lat.values)


---
## 3.3 PDE Residual Map

Residual of the barotropic vorticity equation evaluated from model output:

$$R = \frac{\partial\zeta}{\partial t} + \mathbf{u} \cdot \nabla\zeta$$

If the output satisfies the PDE exactly, $R = 0$ everywhere.
Non-zero residual reveals where the model violates the governing physics.

Computed between consecutive time steps using finite differences.

In [ ]:
plot_advection_residual(ds)


---
## Summary

| # | Visualization | What it shows |
|---|---|---|
| 3.1 | Error Maps | Spatial error structure (abs/signed/relative) |
| 3.2 | Power Spectrum | Spectral fidelity of predictions |
| 3.3 | PDE Residual | Physics consistency of model output |

These diagnostics complement the barotropic-specific visualizations in
`barotropic_visualization.ipynb`.